# 텍스트 특징 추출(TF-IDF)
### 1.샘플 코퍼스로 연습

In [4]:
sample_corpus = [
    '자연어처리 강의를 시작하겠습니다.',
    '자연어처리는 재미있습니다.',
    '밥을 먹고 강의를 듣고 있습니다.',
    '이번 자연어처리 강의는 한국어 자연어처리입니다.'
]

In [3]:
import sklearn

In [6]:
from konlpy.tag import Okt
def my_tokenizer(text):
    return Okt().nouns(text)

In [10]:
# # TfidfVectorizer 객체 생성
# from sklearn.feature_extraction.text import TfidfVectorizer
# vectorizer = TfidfVectorizer(tokenizer=my_tokenizer)
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(tokenizer=my_tokenizer)
# 특징 집합과 관련 데이터 모델을 생성
vectorizer.fit(sample_corpus)
print(vectorizer.get_feature_names_out())
# 특징 벡터 추출
sample_dtm = vectorizer.transform(sample_corpus)
print(sample_dtm.toarray())

['강의' '밥' '시작' '이번' '자연어' '처리' '한국어']
[[1 0 1 0 1 1 0]
 [0 0 0 0 1 1 0]
 [1 1 0 0 0 0 0]
 [1 0 0 1 2 2 1]]


c:\Users\user\AppData\Local\anaconda3\envs\textmine26\lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


### 2. 다음 영화 리뷰로 연습

In [13]:
import pandas as pd
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. 데이터 로드
df = pd.read_csv('D:\Lecture\TextMining26\data\daum_movie_review.csv')

# 데이터 구조 확인 (상위 5개)
print(df.head())

# 'review' 컬럼에 결측치가 있을 경우 제거
df = df.dropna(subset=['review'])

                                              review  rating        date  \
0                             돈 들인건 티가 나지만 보는 내내 하품만       1  2018.10.29   
1       몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.      10  2018.10.26   
2  이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...       8  2018.10.24   
3                                이 정도면 볼만하다고 할 수 있음!       8  2018.10.22   
4                                               재미있다      10  2018.10.20   

    title  
0  인피니티 워  
1  인피니티 워  
2  인피니티 워  
3  인피니티 워  
4  인피니티 워  


In [ ]:
# 2. 토크나이저 함수 정의 (노트북 실습 코드 활용)
okt = Okt()

def my_tokenizer(text):
    # 명사(Noun), 동사(Verb), 형용사(Adjective)만 추출하여 학습 효율을 높입니다.
    selected_words = []
    for word, tag in okt.pos(text, stem=True):
        if tag in ['Noun', 'Verb', 'Adjective']:
            selected_words.append(word)
    return selected_words

# 3. TF-IDF 벡터라이저 객체 생성 및 학습.
vectorizer = TfidfVectorizer(tokenizer=my_tokenizer, min_df=2)

# 전체 리뷰 데이터를 바탕으로 특징 모델 생성 및 변환
reviews = df['review'].tolist()
tfidf_matrix = vectorizer.fit_transform(reviews)

# 추출된 특징(단어) 확인
features = vectorizer.get_feature_names_out()
print(f"추출된 특징 개수: {len(features)}")
print(f"상위 20개 특징: {features[:20]}")

c:\Users\user\AppData\Local\anaconda3\envs\textmine26\lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


추출된 특징 개수: 6580
상위 20개 특징: ['가가' '가게' '가기' '가까이' '가깝다' '가끔' '가나' '가난' '가난하다' '가능' '가능성' '가능하다' '가다'
 '가다가' '가도' '가득' '가득하다' '가디언즈' '가라' '가로']


In [15]:
# '인피니티 워' 영화 리뷰의 TF-IDF 결과 확인 예시
infinity_war_idx = df[df['title'] == '인피니티 워'].index[0]
row = tfidf_matrix.getrow(infinity_war_idx).toarray()[0]

# 해당 리뷰에서 TF-IDF 값이 높은 순으로 정렬
import numpy as np
sorted_indices = np.argsort(row)[::-1]

print(f"\n[{df.iloc[infinity_war_idx]['title']}] 리뷰의 주요 키워드:")
for i in range(10):
    if row[sorted_indices[i]] > 0:
        print(f"{features[sorted_indices[i]]}: {row[sorted_indices[i]]:.4f}")


[인피니티 워] 리뷰의 주요 키워드:
하품: 0.4857
티: 0.4606
들이다: 0.4305
내내: 0.3756
나: 0.3276
돈: 0.3086
보다: 0.1516
